# 01 — Ingestion

Loads the ASSISTments 2012-2013 file (with affect predictions, 35 columns, about 3 GB) and saves only the columns I need as a parquet file.

The file is large mainly because of the number of rows, not one wide column. Reading 12 columns keeps memory manageable, and the full read takes about two minutes.

**Output:** `data/processed/raw_subset.parquet`

In [11]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Project folder and raw CSV location.
EATS = Path(r"C:\Users\vedac\Desktop\EATS")
sys.path.insert(0, str(EATS))

RAW = Path(
    r"C:\Users\vedac\Downloads\2012-2013-data-with-predictions-4-final.csv"
    r"\2012-2013-data-with-predictions-4-final.csv"
)
PROCESSED = EATS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 50)

print("file exists:", RAW.exists())
print(f"size: {RAW.stat().st_size / 1e9:.2f} GB")

file exists: True
size: 3.01 GB


## 1.1 Check the header first

Read a few rows to see the columns and confirm the ones I need exist.

In [12]:
# Read just 5 rows to inspect the schema.
peek = pd.read_csv(RAW, nrows=5, encoding="latin-1", low_memory=False)

print(f"{peek.shape[1]} columns\n")
for i, c in enumerate(peek.columns):
    print(f"{i:>2}  {c}")

35 columns

 0  problem_log_id
 1  skill
 2  problem_id
 3  user_id
 4  assignment_id
 5  assistment_id
 6  start_time
 7  end_time
 8  problem_type
 9  original
10  correct
11  bottom_hint
12  hint_count
13  actions
14  attempt_count
15  ms_first_response
16  tutor_mode
17  sequence_id
18  student_class_id
19  position
20  type
21  base_sequence_id
22  skill_id
23  teacher_id
24  school_id
25  overlap_time
26  template_id
27  answer_id
28  answer_text
29  first_action
30  problemlogid
31  Average_confidence(FRUSTRATED)
32  Average_confidence(CONFUSED)
33  Average_confidence(CONCENTRATING)
34  Average_confidence(BORED)


In [13]:
# How much space does the `actions` column take?
actions_chars = peek["actions"].astype(str).str.len()
other_chars = peek.drop(columns=["actions"]).astype(str).apply(
    lambda r: r.str.len().sum(), axis=1
)

print(f"mean chars in `actions`      : {actions_chars.mean():,.0f}")
print(f"mean chars in all 34 others  : {other_chars.mean():,.0f}")
print(f"\n`actions` is ~{actions_chars.mean() / other_chars.mean():.0f}x "
      f"the size of every other column combined.")
print("\nSample of one `actions` value:")
print(repr(peek['actions'].iloc[0])[:400])

mean chars in `actions`      : 101
mean chars in all 34 others  : 198

`actions` is ~1x the size of every other column combined.

Sample of one `actions` value:
'--- \n- - start\n  - 1348859487561\n  - "959522"\n- - answer\n  - 9852\n  - true\n  - she\n  - \n- - end\n'


## 1.2 Columns used

| Column | Used for |
|---|---|
| `user_id` | student identity |
| `skill`, `skill_id` | knowledge component (BKT is fitted per skill) |
| `problem_id`, `assignment_id` | context |
| `correct` | quiz accuracy |
| `ms_first_response` | response time |
| `hint_count`, `bottom_hint` | hint rate |
| `attempt_count` | error pattern |
| `original` | filters out scaffolding sub-questions |
| `start_time` | used to derive `opportunity` in notebook 02 |

This release has no per-skill practice counter (the 2009-2010 release has one), so the practice order has to be rebuilt from timestamps.

In [14]:
USECOLS = [
    "user_id",
    "skill",
    "skill_id",
    "problem_id",
    "assignment_id",
    "correct",
    "attempt_count",
    "ms_first_response",
    "hint_count",
    "bottom_hint",
    "original",
    "start_time",
]

missing = [c for c in USECOLS if c not in peek.columns]
assert not missing, f"missing from file: {missing}"
print(f"all {len(USECOLS)} required columns present")

all 12 required columns present


## 1.3 Read in chunks

Reading in chunks keeps peak memory lower. `MAX_CHUNKS = 5` reads about a million rows for a quick test; `None` reads the full file.

In [15]:
CHUNKSIZE = 200_000
MAX_CHUNKS = None   # full file (~10M rows, ~2 min). Set to 5 for a quick smoke test.

start = time.time()
chunks = []

reader = pd.read_csv(
    RAW,
    usecols=USECOLS,
    chunksize=CHUNKSIZE,
    encoding="latin-1",
    low_memory=False,
    on_bad_lines="warn",
)

for i, chunk in enumerate(reader):
    if MAX_CHUNKS is not None and i >= MAX_CHUNKS:
        break
    chunks.append(chunk)
    if i % 5 == 0:
        elapsed = time.time() - start
        print(f"  chunk {i:>3}  rows {sum(len(c) for c in chunks):>10,}  "
              f"{elapsed:>6.1f}s", flush=True)

df = pd.concat(chunks, ignore_index=True)
del chunks

print(f"\ndone in {time.time() - start:.1f}s")
print(f"rows: {len(df):,}")
if MAX_CHUNKS is not None:
    print(f"\n*** PARTIAL READ ({MAX_CHUNKS} chunks). Set MAX_CHUNKS = None for the full file. ***")

  chunk   0  rows    200,000     1.6s
  chunk   5  rows  1,200,000     8.7s
  chunk  10  rows  2,200,000    16.0s
  chunk  15  rows  3,200,000    23.2s
  chunk  20  rows  4,200,000    30.8s
  chunk  25  rows  5,200,000    38.3s
  chunk  30  rows  6,123,270    45.3s

done in 45.6s
rows: 6,123,270


## 1.4 Smaller dtypes

Ids fit in 32-bit integers and skill names repeat a lot, so `category` saves memory. Timestamps are parsed here because notebook 02 needs them.

In [16]:
before = df.memory_usage(deep=True).sum()

# Ids and counts: 32-bit integers are far more than enough.
for col in ["user_id", "problem_id", "assignment_id", "skill_id"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

for col in ["correct", "attempt_count", "hint_count", "bottom_hint", "original"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

df["ms_first_response"] = pd.to_numeric(
    df["ms_first_response"], errors="coerce"
).astype("float64")

# Few hundred distinct values across millions of rows - category is a large win.
df["skill"] = df["skill"].astype("category")

# Parse timestamps now: notebook 02 needs them to derive `opportunity`, and datetime64
# is much smaller than the equivalent strings.
df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")

after = df.memory_usage(deep=True).sum()

print(f"before: {before / 1e6:>8,.0f} MB")
print(f"after : {after / 1e6:>8,.0f} MB   ({before / after:.1f}x smaller)")
df.dtypes

before:      776 MB
after :      331 MB   (2.3x smaller)


skill                      category
problem_id                  float32
user_id                     float32
assignment_id               float32
start_time           datetime64[us]
original                    float32
correct                     float32
bottom_hint                 float32
hint_count                  float32
attempt_count               float32
ms_first_response           float64
skill_id                    float32
dtype: object

## 1.5 First summary

A few numbers to check whether the data is usable: skill-tag coverage, timestamp coverage and hint usage.

In [17]:
n = len(df)
summary = pd.DataFrame(
    [
        ("Rows read", f"{n:,}", ""),
        ("Distinct students", f"{df['user_id'].nunique():,}", ""),
        ("Distinct skills", f"{df['skill'].nunique():,}", ""),
        ("Rows with a skill tag", f"{df['skill'].notna().sum():,}",
         f"{df['skill'].notna().mean():.1%}"),
        ("Rows WITHOUT a skill tag", f"{df['skill'].isna().sum():,}",
         "unusable for BKT"),
        ("Main problems (original=1)", f"{(df['original'] == 1).sum():,}",
         f"{(df['original'] == 1).mean():.1%}"),
        ("Valid response times (>0)", f"{(df['ms_first_response'] > 0).sum():,}",
         f"{(df['ms_first_response'] > 0).mean():.1%}"),
        ("Parseable timestamps", f"{df['start_time'].notna().sum():,}",
         f"{df['start_time'].notna().mean():.1%}"),
        ("First-attempt accuracy", f"{df['correct'].mean():.3f}", ""),
        ("Rows with >=1 hint", f"{(df['hint_count'] > 0).sum():,}",
         f"{(df['hint_count'] > 0).mean():.1%}"),
    ],
    columns=["measure", "value", "note"],
)
summary

,measure,value,note
0,Rows read,"6,123,270",
1,Distinct students,"46,674",
2,Distinct skills,198,
3,Rows with a skill tag,"2,630,080",43.0%
4,Rows WITHOUT a skill tag,"3,493,190",unusable for BKT
5,Main problems (original=1),"5,819,737",95.0%
6,Valid response times (>0),"6,123,257",100.0%
7,Parseable timestamps,"6,049,682",98.8%
8,First-attempt accuracy,0.677,
9,Rows with >=1 hint,"846,783",13.8%


### Notes on the summary

- **Skill tags:** rows without a skill can't be used, because BKT is fitted per skill.
- **Timestamps:** `opportunity` is derived by ordering on time, so rows without a time can't be placed in a sequence.
- **Hints:** the hint-rate feature needs enough hint usage to vary.

## 1.6 Duplicate rows

The 2009-2010 release repeats a row for each skill tag on multi-skill problems. I couldn't find whether 2012-2013 does the same, so I check for duplicates on `(user_id, problem_id, start_time)`.

In [18]:
key = ["user_id", "problem_id", "start_time"]
dupes = df.duplicated(subset=key).sum()

print(f"rows sharing (user_id, problem_id, start_time): {dupes:,}  "
      f"({dupes / len(df):.2%})")

if dupes > 0:
    print("\nExample of a duplicated group:")
    d = df[df.duplicated(subset=key, keep=False)].sort_values(key)
    display(d.head(6)[["user_id", "problem_id", "skill", "correct", "start_time"]])
    print("\nIf the rows differ only by `skill`, this is multi-skill tagging and")
    print("must be consolidated in notebook 02.")
else:
    print("\nNo duplicates on that key (the 2009-2010 release does have them).")

rows sharing (user_id, problem_id, start_time): 186  (0.00%)

Example of a duplicated group:


,user_id,problem_id,skill,correct,start_time
467066,119865.0,597837.0,NaN,1.0,NaT
491014,119865.0,597837.0,NaN,0.0,NaT
645967,119865.0,597837.0,NaN,0.0,NaT
1926781,119865.0,597837.0,NaN,0.0,NaT
2380724,119865.0,597837.0,NaN,0.0,NaT
2572117,119865.0,597837.0,NaN,0.0,NaT



If the rows differ only by `skill`, this is multi-skill tagging and
must be consolidated in notebook 02.


## 1.7 Are sequences long enough for BKT?

BKT needs students to practise the same skill several times. The 2009-2010 data is skill-builder data, where sequences are long by design. This file is general assignment data, where a student may see a skill only once.

A partial read cuts student histories short, so this is checked on the full data. If the median sequence length were only 1 or 2, BKT wouldn't be usable here.

In [19]:
# Sequences are only usable when the skill is tagged AND the time is known,
# because `opportunity` is derived by ordering on time.
tagged = df[df["skill"].notna() & df["start_time"].notna()]
seq = tagged.groupby(["user_id", "skill"], observed=True).size()

print(f"usable rows                : {len(tagged):>12,}")
print(f"(student, skill) pairs     : {len(seq):>12,}")
print(f"distinct students          : {tagged['user_id'].nunique():>12,}")
print(f"distinct skills            : {tagged['skill'].nunique():>12,}\n")

print("opportunities per (student, skill):")
print(seq.describe().round(2).to_string())

print("\nsequence length coverage:")
for k in (2, 3, 5, 10, 20):
    print(f"  >= {k:>2} opportunities : {(seq >= k).sum():>9,}  ({(seq >= k).mean():6.1%})")

median = seq.median()
print("\n" + "=" * 58)
if median >= 5:
    print(f"MEDIAN = {median:.0f}  ->  BKT is viable. Continue to notebook 02.")
elif median >= 3:
    print(f"MEDIAN = {median:.0f}  ->  Marginal. Usable if we restrict to longer sequences.")
else:
    print(f"MEDIAN = {median:.0f}  ->  Too short for BKT. Stop and reconsider the dataset.")
print("=" * 58)

usable rows                :    2,579,333
(student, skill) pairs     :      357,516
distinct students          :       28,539
distinct skills            :          198

opportunities per (student, skill):
count    357516.00
mean          7.21
std          11.10
min           1.00
25%           2.00
50%           4.00
75%           8.00
max         335.00

sequence length coverage:
  >=  2 opportunities :   278,012  ( 77.8%)
  >=  3 opportunities :   235,263  ( 65.8%)
  >=  5 opportunities :   158,761  ( 44.4%)
  >= 10 opportunities :    75,344  ( 21.1%)
  >= 20 opportunities :    26,184  (  7.3%)

MEDIAN = 4  ->  Marginal. Usable if we restrict to longer sequences.


### Which skills have usable sequences?

For BKT, the number of students who practised a skill several times matters more than the total number of interactions.

In [20]:
per_skill = (
    tagged.groupby("skill", observed=True)
    .agg(
        interactions=("correct", "size"),
        students=("user_id", "nunique"),
        accuracy=("correct", "mean"),
        hint_rate=("hint_count", lambda s: (s > 0).mean()),
    )
)
per_skill["mean_seq_len"] = per_skill["interactions"] / per_skill["students"]

# How many students practised this skill at least 5 times?
long_enough = (
    seq[seq >= 5].reset_index().groupby("skill", observed=True).size()
)
per_skill["students_with_5plus"] = long_enough
per_skill["students_with_5plus"] = per_skill["students_with_5plus"].fillna(0).astype(int)

candidates = per_skill.sort_values("students_with_5plus", ascending=False).head(20)
print("Top 20 skills by number of students with >= 5 opportunities:\n")
print(candidates.round(3).to_string())

Top 20 skills by number of students with >= 5 opportunities:

                                               interactions  students  accuracy  hint_rate  mean_seq_len  students_with_5plus
skill                                                                                                                        
Addition and Subtraction Integers                    148283     10634     0.722      0.098        13.944                 7539
Addition and Subtraction Fractions                   143881      9733     0.683      0.183        14.783                 7179
Equation Solving Two or Fewer Steps                  191603      8797     0.719      0.110        21.780                 6253
Conversion of Fraction Decimals Percents              95845      8051     0.753      0.131        11.905                 5248
Multiplication and Division Integers                  81806      6989     0.849      0.024        11.705                 4965
Multiplication and Division Positive Decimals         69

**Choosing skills.** I looked for:

1. enough students with five or more attempts on the skill, and
2. mid-range accuracy, so there is room for an intervention to make a difference.

The final selection is made in notebook 02.

## 1.8 Save

In [21]:
OUT = PROCESSED / "raw_subset.parquet"
df.to_parquet(OUT, index=False)

print(f"saved: {OUT}")
print(f"parquet on disk : {OUT.stat().st_size / 1e6:,.0f} MB")
print(f"original CSV    : {RAW.stat().st_size / 1e6:,.0f} MB")
print(f"reduction       : {RAW.stat().st_size / OUT.stat().st_size:.0f}x")

saved: C:\Users\vedac\Desktop\EATS\data\processed\raw_subset.parquet
parquet on disk : 130 MB
original CSV    : 3,009 MB
reduction       : 23x


In [19]:
# Sequences are only usable when the skill is tagged AND the time is known,
# because `opportunity` is derived by ordering on time.
tagged = df[df["skill"].notna() & df["start_time"].notna()]
seq = tagged.groupby(["user_id", "skill"], observed=True).size()

print(f"usable rows                : {len(tagged):>12,}")
print(f"(student, skill) pairs     : {len(seq):>12,}")
print(f"distinct students          : {tagged['user_id'].nunique():>12,}")
print(f"distinct skills            : {tagged['skill'].nunique():>12,}\n")

print("opportunities per (student, skill):")
print(seq.describe().round(2).to_string())

print("\nsequence length coverage:")
for k in (2, 3, 5, 10, 20):
    print(f"  >= {k:>2} opportunities : {(seq >= k).sum():>9,}  ({(seq >= k).mean():6.1%})")

median = seq.median()
print("\n" + "=" * 58)
if median >= 5:
    print(f"MEDIAN = {median:.0f}  ->  BKT is viable. Continue to notebook 02.")
elif median >= 3:
    print(f"MEDIAN = {median:.0f}  ->  Marginal. Usable if we restrict to longer sequences.")
else:
    print(f"MEDIAN = {median:.0f}  ->  Too short for BKT. Stop and reconsider the dataset.")
print("=" * 58)

usable rows                :    2,579,333
(student, skill) pairs     :      357,516
distinct students          :       28,539
distinct skills            :          198

opportunities per (student, skill):
count    357516.00
mean          7.21
std          11.10
min           1.00
25%           2.00
50%           4.00
75%           8.00
max         335.00

sequence length coverage:
  >=  2 opportunities :   278,012  ( 77.8%)
  >=  3 opportunities :   235,263  ( 65.8%)
  >=  5 opportunities :   158,761  ( 44.4%)
  >= 10 opportunities :    75,344  ( 21.1%)
  >= 20 opportunities :    26,184  (  7.3%)

MEDIAN = 4  ->  Marginal. Usable if we restrict to longer sequences.


In [20]:
per_skill = (
    tagged.groupby("skill", observed=True)
    .agg(
        interactions=("correct", "size"),
        students=("user_id", "nunique"),
        accuracy=("correct", "mean"),
        hint_rate=("hint_count", lambda s: (s > 0).mean()),
    )
)
per_skill["mean_seq_len"] = per_skill["interactions"] / per_skill["students"]

# How many students practised this skill at least 5 times?
long_enough = (
    seq[seq >= 5].reset_index().groupby("skill", observed=True).size()
)
per_skill["students_with_5plus"] = long_enough
per_skill["students_with_5plus"] = per_skill["students_with_5plus"].fillna(0).astype(int)

candidates = per_skill.sort_values("students_with_5plus", ascending=False).head(20)
print("Top 20 skills by number of students with >= 5 opportunities:\n")
print(candidates.round(3).to_string())

Top 20 skills by number of students with >= 5 opportunities:

                                               interactions  students  accuracy  hint_rate  mean_seq_len  students_with_5plus
skill                                                                                                                        
Addition and Subtraction Integers                    148283     10634     0.722      0.098        13.944                 7539
Addition and Subtraction Fractions                   143881      9733     0.683      0.183        14.783                 7179
Equation Solving Two or Fewer Steps                  191603      8797     0.719      0.110        21.780                 6253
Conversion of Fraction Decimals Percents              95845      8051     0.753      0.131        11.905                 5248
Multiplication and Division Integers                  81806      6989     0.849      0.024        11.705                 4965
Multiplication and Division Positive Decimals         69